# VisionWeave: spending visual tokens like money

Recomputes the headline numbers of VisionWeave (arXiv:2610.07987, Feng et al., Oct 6 2026) from the paper's Table 4 (Qwen3.8-27B, 512-token budget) and Table 5 (SGLang serving), plus toy demos of the gated pooler and the attention-cost scaling. All numbers below are the paper's reported figures; the code only re-derives the aggregates.

In [ ]:
# Table 4, Qwen3.8-27B: native score, VisionWeave score, token savings (%)
bench = ["RealWorldQA","DocVQA","InfoVQA","ScreenSpotV2","HallusionBench","VideoOCR","Video-MME","LongVideoBench"]
native = [78.82, 92.47, 74.13, 94.42, 73.25, 70.72, 70.89, 66.49]
weave  = [77.91, 91.33, 71.61, 91.43, 73.60, 70.51, 71.04, 66.42]
sav    = [43.1, 27.3, 16.5, 55.1, 44.9, 49.6, 54.2, 53.0]
import statistics
mean_sav = statistics.mean(sav)
ret_per_bench = [w/n*100 for w,n in zip(weave, native)]
mean_ret = statistics.mean(ret_per_bench)
avg_ret = sum(weave)/sum(native)*100
print(f"mean savings = {mean_sav:.2f}%  (paper: 43.0%)")
print(f"mean per-benchmark retention = {mean_ret:.2f}%  (paper: 98.9%)")
print(f"average-score retention = {avg_ret:.2f}%")
print(f"average loss = {statistics.mean(native)-statistics.mean(weave):.2f} points; mean per-benchmark % loss = {statistics.mean([(n-w)/n*100 for n,w in zip(native,weave)]):.2f}% (paper: 1.08%)")
assert abs(mean_sav-43.0) < 0.1 and abs(mean_ret-98.9) < 0.1

In [ ]:
# Tokens remaining from the 512-token budget
for b, s in zip(bench, sav):
    print(f"{b:15s} saves {s:4.1f}%  ->  {512*(1-s/100):5.0f} tokens remain")
avg_remain = 512*(1-mean_sav/100)
print(f"average remaining: {avg_remain:.0f} tokens (vs 256 under a fixed 50% cut)")
assert 290 < avg_remain < 293

In [ ]:
# Toy gated pooler vs mean pooling: one sharp detail patch among three flat ones
import numpy as np
detail = np.array([10.0, 0.0]); flat = np.array([0.0, 1.0])
patches = [detail, flat, flat, flat]
mean_pool = np.mean(patches, axis=0)
# gate weights ~ softmax of patch norms (content-dependent, as in the paper's gate)
norms = np.array([np.linalg.norm(p) for p in patches])
w = np.exp(norms)/np.exp(norms).sum()
gated = sum(wi*p for wi, p in zip(w, patches))
print("gate weights:", np.round(w, 4))
print("mean pool :", mean_pool, " dist to detail:", round(np.linalg.norm(mean_pool-detail), 3))
print("gated pool:", np.round(gated, 3), " dist to detail:", round(np.linalg.norm(gated-detail), 3))
assert np.linalg.norm(gated-detail) < np.linalg.norm(mean_pool-detail)
assert w[0] > 0.99  # the detail patch dominates the gate, unlike the 0.25 mean weight

In [ ]:
# Why attention cost falls faster than the token count (illustrative, O(n^2))
n0, n1 = 512, 292  # 292 = 512 * (1 - 0.43)
ratio = (n1/n0)**2
print(f"vision attention pairs: {n0**2} -> {n1**2}  ({ratio:.3f}x, i.e. {(1-ratio)*100:.1f}% fewer)")
assert abs(ratio-0.325) < 0.01

In [ ]:
# ScreenSpotV2 collapse: pruning vs weaving (Table 4, Qwen3.8-27B)
for name, score in [("Native",94.42),("VisionWeave",91.43),("FastV fixed-50%",49.53),("VisionZip fixed-50%",49.21)]:
    print(f"{name:20s} {score:5.2f}  ({score-94.42:+.2f} vs native)")
# router shuffle ablation (Table 6): same budget, wrong places
print("Shuffled-router average: 76.73 -> 71.37 (-5.36); ScreenSpotV2 alone: 91.43 -> 59.28 (-32.15)")
assert 94.42-91.43 < 94.42-49.53  # weaving loses far less than pruning

In [ ]:
# Table 5: SGLang serving, Qwen3.8-27B, long-video workload
thr0, thr1 = 0.90, 2.07
ttft0, ttft1 = 169.90, 77.54
tpot0, tpot1 = 355.64, 140.29
print(f"throughput: {thr1/thr0:.2f}x (paper: 2.30x)")
print(f"mean TTFT reduction: {(1-ttft1/ttft0)*100:.1f}% (paper: 54.4%)")
print(f"mean TPOT reduction: {(1-tpot1/tpot0)*100:.1f}% (paper: 60.6%)")
assert abs(thr1/thr0-2.30) < 0.01
assert abs((1-ttft1/ttft0)*100-54.4) < 0.2
assert abs((1-tpot1/tpot0)*100-60.6) < 0.2